In [ ]:
import pandas as pd

In [ ]:

df = pd.read_excel('../Dataset 10000/subdataset.xlsx')

In [ ]:
caminho_salvar = '/home/cecilia/Documentos/PIBIC/Fase3/Resultados_Rotulagem_Parciais/gemma_parcial.json'


In [26]:
import os
os.makedirs(os.path.dirname(caminho_salvar), exist_ok=True)

In [ ]:
import shutil
from openai import OpenAI
import pandas as pd
from typing import List, Dict, Any
import time
from tqdm.notebook import tqdm
import os
import json


class OpenRouterBatchInference:
    def __init__(self, api_key: str, models: List[str], system_prompt: str):
      
        self.client = OpenAI(
            base_url="",
            api_key='ollama'
        )
        self.models = models
        self.system_prompt = system_prompt

    def _create_messages(self, user_prompt: str) -> List[Dict[str, str]]:
        
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Input: {user_prompt}\nOutput:"}
        ]

    def _query_model(self, model: str, user_prompt: str) -> str:
        
        completion = self.client.chat.completions.create(
            model=model,
            messages=self._create_messages(user_prompt),
            timeout=120,
            max_tokens=500
        )

        msg = completion.choices[0].message
        finish_reason = completion.choices[0].finish_reason

        if finish_reason == "length":
            print("Resposta cortada por limite de tokens!")

        if msg.content:
            return msg.content

        if hasattr(msg, 'reasoning') and msg.reasoning:
            return msg.reasoning

        return "ERRO: output vazio"

    def generate_outputs(self, dataset: pd.DataFrame, save_path: str) -> Dict[str, List[Dict[str, Any]]]:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        all_outputs = {model: [] for model in self.models}

        if os.path.exists(save_path):
            with open(save_path, 'r', encoding='utf-8') as f:
                all_outputs = json.load(f)
                print("Progresso anterior carregado com sucesso!")

        print("Montando os índices já processados na memória... Aguarde.")

        indices_processados = {
            model: set(int(item["index"]) for item in all_outputs[model] if "index" in item)
            for model in self.models
        }

        loop_progresso = tqdm(dataset.index, desc="Processando dataset")

        for index in loop_progresso:
            data_point = dataset.loc[index]
            user_prompt = f"Input: {data_point['texto_cliente']}"

            if all(int(index) in indices_processados[model] for model in self.models):
                continue

            for model in self.models:
                if int(index) in indices_processados[model]:
                    continue

                try:
                    loop_progresso.set_postfix(modelo=model)
                    output = self._query_model(model, user_prompt)

                    print(f"\n--- Reclamação {index} ---")
                    print(output)
                    print("-" * 50)


                    all_outputs[model].append({
                        "index": int(index),
                        "input": data_point.to_dict(),
                        "output": output
                    })

                except Exception as e:
                    print(f"\n[Erro] no modelo {model} no índice {index}: {e}")
                    all_outputs[model].append({
                        "index": int(index),
                        "input": data_point.to_dict(),
                        "output": "null|Error#API"
                    })

            if (int(index) + 1) % 50 == 0:
                with open(save_path, 'w', encoding='utf-8') as f:
                    json.dump(all_outputs, f, ensure_ascii=False, indent=4)

            if (int(index) + 1) % 1000 == 0:
                caminho_backup = save_path.replace(".json", f"_backup_{int(index)+1}.json")
                with open(caminho_backup, 'w', encoding='utf-8') as f:
                    json.dump(all_outputs, f, ensure_ascii=False, indent=4)

                print(f"Backup salvo: {caminho_backup}")

        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(all_outputs, f, ensure_ascii=False, indent=4)

        return all_outputs

In [ ]:

api_key = 'ollama'
models = [
  'gemma3:27b'
]


In [ ]:
system_prompt = (
"""
Persona:
Você é um especialista em análise de reclamações online, especificamente no CRM, responsável por desenvolver estratégias de negócios, com foco em entender as necessidades do consumidor.

Contexto:
    - O dataset contém reclamações extensas de consumidores online sobre diversos domínios
    - Cada reclamação contém pelo menos um alvo principal específico (podem ter mais de um), correspondente à frase específica a qual o cliente expressa o problema mais relevante da reclamação.
    - Termo de aspecto: Atributo específico ao qual a frase se refere.
    - Categoria do aspecto: Par no formato Entidade#Atributo, onde o 1º refere-se ao elemento da reclamação, e o 2º representa a dimensão que esta sendo avaliada.


Tarefa:
Você deve extrair de cada reclamação, um par contendo o aspecto e sua categoria.
Para isso você deve seguir os passos:
1- Encontre dentro da reclamação o alvo principal, frase a qual contém o problema central da reclamação. (podendo haver mais de um alvo)
2- Dentro do alvo (para cada alvo), encontre um par contendo o termo de aspecto e a sua respectiva categoria
3- Retorne na saída o alvo encontrado e embaixo o par solicitado.

Siga o modelo abaixo para a saída:
	Alvo principal: o alvo principal da reclamação,
	Rótulo (s): aspecto|CATEGORIA

#########Atenção######:

Quando existir mais de um alvo por reclamação, retorne cada um com seu respectivo par abaixo, como no modelo a seguir:
        Alvo 1: "1º alvo que você encontrar"
        Rótulo(s): {{aspecto|CATEGORIA}}
        
        Alvo 2: "2º alvo que você encontrar"
        Rótulo(s): {{aspecto|CATEGORIA}}

Quando existir mais de um aspecto e categoria para cada alvo encontrado, retorne cada par dentro de '{{}}' específicas, sendo que cada par deve estar separado por uma ',' e com sua numeração específica. Siga este modelo: 
	Rótulo(s): 1:{{aspecto|CATEGORIA}}, 2: {{aspecto|CATEGORIA}}

Quando o aspecto não estiver explicitamente escrito no texto, infera-o pelo contexto e escreva o Aspecto inferido e o termo 'implícito' entre '()'. Como no exemplo a seguir:
	Rótulo(s): Aspecto (implícito)|CATEGORIA
	
Se não for possível inferir nenhum aspecto, retorne somente a palavra 'implícito' no lugar do aspecto e a sua respectiva categoria: 
	Rótulo(s): Implícito|CATEGORIA


#######Observações########:
- Separe cada par por uma barra vertical como esta: ‘|’
- Não extraia aspectos mencionados apenas como contexto, histórico ou consequência do problema principal. Priorize sempre o aspecto diretamente associado à reclamação central do consumidor.
- Não extraia aspectos secundários ou periféricos.
- Retorne somente as categorias em caixa alta. O termo de aspecto deve manter a capitalização natural do texto.
- Não utilize aspas (retas, curvas ou de qualquer tipo) ao redor do alvo ou do texto extraído. Escreva o texto diretamente, sem aspas envolvendo.


##########EXEMPLOS##############

#####Exemplo1####

Reclamação completa:
“Boa tarde Empréstimo consignado já foi pago documentos e contrato não consiste com vício documentos peço o cancelamento do contrato do banco inter”

Alvo principal:  Empréstimo consignado já foi pago
Rótulo(s): contrato|CONTRATO#CANCELAMENTO

####Exemplo2#######

Reclamação completa:
“Após o recebimento de ligação de cobrança, fui induzido a ingressar na plataforma do SERASA LIMPA NOME, na qual pude constatar a existência de um débito em meu nome no valor de R$ 3.084,11, correspondente ao contrato nº 1662873 débito este não reconhecido pelo NOTIFICANTE.”

Alvo 1: constatar a existência de um débito em meu nome
Rótulo(s): débito|COBRANÇA#INDEVIDA

Alvo 2: débito este não reconhecido pelo NOTIFICANTE
Rótulo(s): débito|COBRANÇA#RECONHECIMENTO

######Exemplo3########


Reclamação completa:

"Estou tentando solicitar um novo cartão de débito para o endereço Rua dos Guenoas, 1088, casa 3, Guarujá (91770060). POA/RS mas o serviço de telefone diz que devo procurar o chat e o chat diz que devo procurar o telefone. Não posso usar meu cartão de débito porque ninguém resolve e fica jogando o problema de um para o outro."

Alvo principal: o serviço de telefone diz que devo procurar o chat e o chat diz que devo procurar o telefone. Não posso usar meu cartão de débito porque ninguém resolve e fica jogando o problema de um para o outro

Rótulo(s): Atendimento(implícito)|ATENDIMENTO#CANAL

"""
)


inference = OpenRouterBatchInference(
    api_key=api_key,
    models=models,
    system_prompt=system_prompt
)



In [36]:
modelo = models[0]


In [ ]:
import os

if os.path.exists(caminho_teste):
    os.remove(caminho_teste)
    print("Arquivo de teste antigo apagado, começando do zero.")
else:
    print("Nenhum arquivo anterior encontrado.")



In [ ]:
df_teste = df.head(100) 

caminho_teste = '/home/cecilia/Documentos/PIBIC/Fase3/Resultados_Rotulagem_Parciais/gemma_teste100.json'

outputs_teste = inference.generate_outputs(df_teste, caminho_teste)

In [ ]:
caminho_completo = '/home/cecilia/Documentos/PIBIC/Fase3/Resultados_Rotulagem_Parciais/gemma_completo (1).json'

In [ ]:
import os

if os.path.exists(caminho_completo):
    os.remove(caminho_completo)
    print("Arquivo de teste antigo apagado, começando do zero.")
else:
    print("Nenhum arquivo anterior encontrado.")


In [ ]:
outputs_completo = inference.generate_outputs(df, caminho_completo)

In [ ]:
import re

def limpar_aspas(texto):
    if texto is None:
        return texto
    return texto.strip().strip('"').strip("'").strip('"').strip('"').strip()

def parse_output(texto):
    pares = []
    blocos = re.split(r'(?=Alvo\s*(?:principal)?\s*\d*\s*[:"])', texto, flags=re.IGNORECASE)

    for bloco in blocos:
        bloco = bloco.strip()
        if not bloco:
            continue

        alvo_match = re.search(r'Alvo[^:]*:\s*"?([^"\n]+)"?', bloco, flags=re.IGNORECASE)
        alvo = limpar_aspas(alvo_match.group(1)) if alvo_match else None

        rotulo_match = re.search(r'R[oó]tulo\(?s?\)?:?\s*"?(.+)', bloco, flags=re.IGNORECASE | re.DOTALL)
        rotulo = limpar_aspas(rotulo_match.group(1)) if rotulo_match else None

        if alvo and rotulo:
            pares.append((alvo, rotulo))

    return pares

In [ ]:
import json
import pandas as pd

with open(caminho_completo, 'r', encoding='utf-8') as f:
    outputs_completo = json.load(f)

modelo = models[0]

linhas = []
for item in outputs_completo[modelo]:
    frase = item["input"]["texto_cliente"]  
    saida = item["output"]
    pares = parse_output(saida)  

    if pares:
        for alvo, rotulo in pares:
            linhas.append({
                "Frase_Original": frase,
                "Alvo": alvo,
                "Rótulo": rotulo
            })
    else:
        linhas.append({
            "Frase_Original": frase,
            "Alvo": None,
            "Rótulo": saida
        })

df_final = pd.DataFrame(linhas)

df_final.to_excel("/home/cecilia/Documentos/PIBIC/Dados_subdataset_rotulado/subdataset_rotulado.xlsx", index=False)
print(f"Salvo! {len(df_final)} linhas geradas.")